In [3]:
# Cell 1: upload and load the data
# Code written with assistance from Claude (Anthropic) and reviewed by the author.
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(filename)
print(df.shape)        # expect (9804, 20)

Saving openfoodfacts_nutrition_final_2025-12-10.csv to openfoodfacts_nutrition_final_2025-12-10 (1).csv
openfoodfacts_nutrition_final_2025-12-10 (1).csv
(9804, 20)


In [4]:
# Cell 2: clean and split
from sklearn.model_selection import train_test_split

cols = ["fat_100g", "saturated-fat_100g", "sugars_100g", "salt_100g"]
d = df.dropna(subset=["nova_group"] + cols).copy()
print("With NOVA label and all four nutrients:", len(d))   # expect 8376

d["is_upf"] = d["nova_group"] == 4
bad = (d[cols] > 100).any(axis=1) | (d["saturated-fat_100g"] > d["fat_100g"])
d = d[~bad].copy()
print("After removing impossible rows:", len(d))           # expect 8366

dev, test = train_test_split(d, test_size=0.3, random_state=42, stratify=d["is_upf"])
print("Development:", len(dev), "| Test:", len(test))      # expect 5856 and 2510

With NOVA label and all four nutrients: 8376
After removing impossible rows: 8366
Development: 5856 | Test: 2510


In [5]:
# Cell 3: Algorithm A (UK thresholds per 100 g; Department of Health, 2016)
LIMITS = {"fat_100g": (3.0, 17.5), "saturated-fat_100g": (1.5, 5.0),
          "sugars_100g": (5.0, 22.5), "salt_100g": (0.3, 1.5)}

def colour(value, low, high):
    return "green" if value <= low else "amber" if value <= high else "red"

def algorithm_a(row, k):
    reds = [colour(row[n], lo, hi) for n, (lo, hi) in LIMITS.items()].count("red")
    return reds >= k      # True means predicted ultra-processed

# Check against Worked Example A in the guidance
example = {"fat_100g": 2.2, "saturated-fat_100g": 0.4, "sugars_100g": 1.5, "salt_100g": 0.35}
print([colour(example[n], lo, hi) for n, (lo, hi) in LIMITS.items()])
# expect ['green', 'green', 'green', 'amber']

['green', 'green', 'green', 'amber']


In [6]:
# Cell 4: Algorithm A results on the development set (the test set is NOT used here)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef)

y = dev["is_upf"]
print("Baseline (predict all ultra-processed):", round(y.mean(), 3))
for k in [1, 2, 3, 4]:
    pred = dev.apply(lambda r: algorithm_a(r, k), axis=1)
    print(k, "| acc", round(accuracy_score(y, pred), 3),
          "| prec", round(precision_score(y, pred, zero_division=0), 3),
          "| recall", round(recall_score(y, pred), 3),
          "| F1", round(f1_score(y, pred), 3),
          "| MCC", round(matthews_corrcoef(y, pred), 3))

Baseline (predict all ultra-processed): 0.594
1 | acc 0.531 | prec 0.624 | recall 0.53 | F1 0.573 | MCC 0.061
2 | acc 0.48 | prec 0.625 | recall 0.314 | F1 0.418 | MCC 0.041
3 | acc 0.474 | prec 0.807 | recall 0.151 | F1 0.255 | MCC 0.153
4 | acc 0.406 | prec 1.0 | recall 0.001 | F1 0.002 | MCC 0.022


In [7]:
# Cell 5: error analysis on the development set only
from sklearn.metrics import confusion_matrix

pred1 = dev.apply(lambda r: algorithm_a(r, 1), axis=1)
print(confusion_matrix(dev["is_upf"], pred1))
# rows = actual (not UPF, UPF); columns = predicted (not UPF, UPF)

extra = ["energy-kcal_100g", "proteins_100g", "fiber_100g"]
print(dev.groupby("is_upf")[cols + extra].median().round(2))

print("\nUltra-processed foods with NO red light (missed):")
print(dev[dev["is_upf"] & ~pred1][["product_name"] + cols].sample(10, random_state=1).to_string())

print("\nNot ultra-processed foods WITH a red light (false alarms):")
print(dev[~dev["is_upf"] & pred1][["product_name"] + cols].sample(10, random_state=1).to_string())

[[1263 1112]
 [1636 1845]]
        fat_100g  saturated-fat_100g  sugars_100g  salt_100g  \
is_upf                                                         
False       5.75                1.20          2.8       0.26   
True        8.80                1.71          6.0       0.50   

        energy-kcal_100g  proteins_100g  fiber_100g  
is_upf                                               
False              287.0            6.6        2.80  
True               284.5            6.3        3.08  

Ultra-processed foods with NO red light (missed):
                                        product_name  fat_100g  saturated-fat_100g  sugars_100g  salt_100g
5337                     Ristorante Pizza Mozzarella      12.0                3.83         3.35   0.942000
2323                           Confitura de Fresa 0%       0.0                0.00         3.40   0.000000
6744                  PET 1,25L LORINA LIMONADE DOUB       0.0                0.00         3.90   0.010000
389                  

In [8]:
# Cell 6: quantify the failure patterns (development set only; exploratory cut-offs)
red_df = pd.DataFrame({n: dev[n] > hi for n, (lo, hi) in LIMITS.items()})
print("Share of foods with a red, by nutrient and class:")
print(red_df.groupby(dev["is_upf"]).mean().round(3))

only_fat = (red_df["fat_100g"] | red_df["saturated-fat_100g"]) & ~red_df["sugars_100g"] & ~red_df["salt_100g"]
print("\nReds from fat/saturates only -> not UPF:", (only_fat & ~dev["is_upf"]).sum(),
      "| UPF:", (only_fat & dev["is_upf"]).sum())

ingredient_like = (dev["fat_100g"] >= 80) | (dev["salt_100g"] >= 50) | (dev["sugars_100g"] >= 80)
print("Ingredient-like foods:", ingredient_like.sum(), "| of which UPF:", (ingredient_like & dev["is_upf"]).sum())

nothing = (dev["fat_100g"] <= 1) & (dev["sugars_100g"] <= 5) & (dev["salt_100g"] <= 0.3)
print("Very low fat, sugar and salt:", nothing.sum(), "| of which UPF:", (nothing & dev["is_upf"]).sum())

Share of foods with a red, by nutrient and class:
        fat_100g  saturated-fat_100g  sugars_100g  salt_100g
is_upf                                                      
False      0.325               0.284        0.082      0.107
True       0.335               0.300        0.241      0.121

Reds from fat/saturates only -> not UPF: 677 | UPF: 623
Ingredient-like foods: 203 | of which UPF: 19
Very low fat, sugar and salt: 421 | of which UPF: 215


In [9]:
# Cell 7: how informative is each colour? Candidate exclusion rules (development set only)
cl = pd.DataFrame({n: dev[n].apply(lambda v, lo=lo, hi=hi: colour(v, lo, hi))
                   for n, (lo, hi) in LIMITS.items()})
y = dev["is_upf"]

for n in LIMITS:
    t = y.groupby(cl[n]).agg(["mean", "size"]).round(3)
    t.columns = ["share_UPF", "n_foods"]
    print(n)
    print(t.reindex(["green", "amber", "red"]))
    print()

all_green = (cl == "green").all(axis=1)
print("All four green:", all_green.sum(), "| share UPF:", round(y[all_green].mean(), 3), "\n")

# Start from "ultra-processed" and exclude natural-looking foods, step by step
base = pd.Series(True, index=dev.index)
step1 = base & ~ingredient_like
step2 = step1 & ~only_fat
step3 = step2 & ~all_green
for name, p in [("all ultra-processed (baseline)", base),
                ("minus ingredient-like", step1),
                ("minus fat-only reds as well", step2),
                ("minus all-green foods as well", step3)]:
    print(name, "| acc", round(accuracy_score(y, p), 3),
          "| MCC", round(matthews_corrcoef(y, p), 3))

fat_100g
          share_UPF  n_foods
fat_100g                    
green         0.534     2167
amber         0.661     1753
red           0.602     1936

saturated-fat_100g
                    share_UPF  n_foods
saturated-fat_100g                    
green                   0.567     2939
amber                   0.643     1200
red                     0.607     1717

sugars_100g
             share_UPF  n_foods
sugars_100g                    
green            0.492     3324
amber            0.672     1499
red              0.812     1033

salt_100g
           share_UPF  n_foods
salt_100g                    
green          0.541     2681
amber          0.644     2501
red            0.625      674

All four green: 729 | share UPF: 0.447 

all ultra-processed (baseline) | acc 0.594 | MCC 0.0
minus ingredient-like | acc 0.623 | MCC 0.193
minus fat-only reds as well | acc 0.611 | MCC 0.145
minus all-green foods as well | acc 0.624 | MCC 0.206


In [10]:
# Cell 8: candidate rule sets and a non-green count (development set only)
sets = {
    "baseline: all ultra-processed": base,
    "minus ingredient-like": base & ~ingredient_like,
    "minus ingredient-like and all-green": base & ~ingredient_like & ~all_green,
}
for name, p in sets.items():
    print(name, "| acc", round(accuracy_score(y, p), 3),
          "| prec", round(precision_score(y, p), 3),
          "| recall", round(recall_score(y, p), 3),
          "| MCC", round(matthews_corrcoef(y, p), 3))

non_green = (cl != "green").sum(axis=1)
t = y.groupby(non_green).agg(["mean", "size"]).round(3)
t.columns = ["share_UPF", "n_foods"]
print("\nShare ultra-processed by number of non-green nutrients:")
print(t)

sugar_red = cl["sugars_100g"] == "red"
print("\nSugars red:", sugar_red.sum(), "| share UPF:", round(y[sugar_red].mean(), 3))
print("Sugars red and not ingredient-like:", (sugar_red & ~ingredient_like).sum(),
      "| share UPF:", round(y[sugar_red & ~ingredient_like].mean(), 3))

baseline: all ultra-processed | acc 0.594 | prec 0.594 | recall 1.0 | MCC 0.0
minus ingredient-like | acc 0.623 | prec 0.612 | recall 0.995 | MCC 0.193
minus ingredient-like and all-green | acc 0.636 | prec 0.637 | recall 0.901 | MCC 0.199

Share ultra-processed by number of non-green nutrients:
   share_UPF  n_foods
0      0.447      729
1      0.551     1290
2      0.526     1195
3      0.626     1935
4      0.853      707

Sugars red: 1033 | share UPF: 0.812
Sugars red and not ingredient-like: 990 | share UPF: 0.841


In [11]:
# Cell 9: Algorithm B - a small forward-chaining engine (development set only)
# Code written with assistance from Claude (Anthropic) and reviewed by the author.
SHORT = {"fat": "fat_100g", "sat": "saturated-fat_100g",
         "sugars": "sugars_100g", "salt": "salt_100g"}
PURE = {"fat": 80, "salt": 50, "sugars": 80}   # cut-offs chosen on the development set

# Each rule: (list of conditions, fact to add)
RULES = []
for n in SHORT:
    RULES += [([f"{n}:amber"], f"{n}:non_green"), ([f"{n}:red"], f"{n}:non_green")]
RULES += [
    (["fat:pure"], "ingredient_like"),
    (["salt:pure"], "ingredient_like"),
    (["sugars:pure"], "ingredient_like"),
    ([f"{n}:green" for n in SHORT], "all_green"),
    ([f"{n}:non_green" for n in SHORT], "all_non_green"),
    (["ingredient_like"], "exclude"),
    (["all_green"], "exclude"),
]

def observe(row):
    facts = set()
    for short, col in SHORT.items():
        lo, hi = LIMITS[col]
        facts.add(f"{short}:{colour(row[col], lo, hi)}")
    for short, cut in PURE.items():
        if row[SHORT[short]] >= cut:
            facts.add(f"{short}:pure")
    return facts

def forward_chain(facts, rules):
    facts = set(facts)
    changed = True
    while changed:                      # repeat until no rule adds a new fact
        changed = False
        for conditions, conclusion in rules:
            if conclusion not in facts and all(c in facts for c in conditions):
                facts.add(conclusion)
                changed = True
    return facts

def algorithm_b(row):
    return "exclude" not in forward_chain(observe(row), RULES)   # True = ultra-processed

# Worked example: butter-like food (fat 82, saturates 57, sugars 1, salt 0.03)
print(sorted(forward_chain(observe({"fat_100g": 82, "saturated-fat_100g": 57,
                                    "sugars_100g": 1, "salt_100g": 0.03}), RULES)))

# Check: B should match the flat rule from Cell 8 exactly
pred_b = dev.apply(algorithm_b, axis=1)
flat = base & ~ingredient_like & ~all_green
print("B agrees with the flat rules on every food:", bool((pred_b == flat).all()))

print("B | acc", round(accuracy_score(y, pred_b), 3),
      "| prec", round(precision_score(y, pred_b), 3),
      "| recall", round(recall_score(y, pred_b), 3),
      "| F1", round(f1_score(y, pred_b), 3),
      "| MCC", round(matthews_corrcoef(y, pred_b), 3))

tier = dev.apply(lambda r: "all_non_green" in forward_chain(observe(r), RULES), axis=1)
print("All four non-green:", int(tier.sum()), "| precision:", round(y[tier].mean(), 3),
      "| recall:", round(tier[y].mean(), 3))

['exclude', 'fat:non_green', 'fat:pure', 'fat:red', 'ingredient_like', 'salt:green', 'sat:non_green', 'sat:red', 'sugars:green']
B agrees with the flat rules on every food: True
B | acc 0.636 | prec 0.637 | recall 0.901 | F1 0.746 | MCC 0.199
All four non-green: 707 | precision: 0.853 | recall: 0.173


In [12]:
# Cell 10: FINAL evaluation on the test set - run once only. Rules are frozen.
# Code written with assistance from Claude (Anthropic) and reviewed by the author.
import numpy as np
from sklearn.metrics import confusion_matrix

yt = test["is_upf"]
pt_base = pd.Series(True, index=test.index)
pt_a = test.apply(lambda r: algorithm_a(r, 3), axis=1)     # k = 3, chosen on dev by MCC
pt_b = test.apply(algorithm_b, axis=1)

for name, p in [("baseline", pt_base), ("A (k=3)", pt_a), ("B", pt_b)]:
    print(name, "| acc", round(accuracy_score(yt, p), 3),
          "| prec", round(precision_score(yt, p, zero_division=0), 3),
          "| recall", round(recall_score(yt, p), 3),
          "| F1", round(f1_score(yt, p), 3),
          "| MCC", round(matthews_corrcoef(yt, p), 3))

print("\nConfusion matrix A (rows actual not-UPF/UPF; columns predicted):")
print(confusion_matrix(yt, pt_a))
print("Confusion matrix B:")
print(confusion_matrix(yt, pt_b))

tier_t = test.apply(lambda r: "all_non_green" in forward_chain(observe(r), RULES), axis=1)
print("\nAll four non-green tier | n:", int(tier_t.sum()),
      "| precision:", round(yt[tier_t].mean(), 3),
      "| recall:", round(tier_t[yt].mean(), 3))

# Bootstrap: 95% interval for B's accuracy gain over the baseline
rng = np.random.default_rng(42)
y_arr, b_arr, base_arr = yt.to_numpy(), pt_b.to_numpy(), pt_base.to_numpy()
diffs = []
for _ in range(1000):
    s = rng.choice(len(y_arr), size=len(y_arr), replace=True)
    diffs.append((b_arr[s] == y_arr[s]).mean() - (base_arr[s] == y_arr[s]).mean())
print("\nB minus baseline accuracy, 95% interval:", np.percentile(diffs, [2.5, 97.5]).round(3))

baseline | acc 0.594 | prec 0.594 | recall 1.0 | F1 0.746 | MCC 0.0
A (k=3) | acc 0.485 | prec 0.816 | recall 0.172 | F1 0.284 | MCC 0.171
B | acc 0.642 | prec 0.643 | recall 0.895 | F1 0.748 | MCC 0.217

Confusion matrix A (rows actual not-UPF/UPF; columns predicted):
[[ 960   58]
 [1235  257]]
Confusion matrix B:
[[ 277  741]
 [ 157 1335]]

All four non-green tier | n: 308 | precision: 0.834 | recall: 0.172

B minus baseline accuracy, 95% interval: [0.033 0.064]


In [13]:
# Cell 11: confirm B equals the flat equivalent on the test set (no labels used)
ing_t = (test["fat_100g"] >= 80) | (test["salt_100g"] >= 50) | (test["sugars_100g"] >= 80)
cl_t = pd.DataFrame({n: test[n].apply(lambda v, lo=lo, hi=hi: colour(v, lo, hi))
                     for n, (lo, hi) in LIMITS.items()})
flat_t = ~ing_t & ~(cl_t == "green").all(axis=1)
print("B equals the flat rules on every test food:", bool((pt_b == flat_t).all()))

B equals the flat rules on every test food: True
